In [0]:
from pyspark.sql import functions as F

# 1. Ajuste o caminho para o seu Volume
path_base = "/Volumes/workspace/default/e-commerce/"

# 2. Criar o database bronze caso ele não exista
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

print("Ambiente configurado e Database 'bronze' pronto.")

Ambiente configurado e Database 'bronze' pronto.


In [0]:
def carregar_csv_para_bronze(nome_arquivo, nome_tabela):
    path_completo = f"{path_base}{nome_arquivo}"
    
    print(f"Processando arquivo: {nome_arquivo}")
    
    # Lendo o CSV
    df = (spark.read
          .format("csv")
          .option("header", "true")
          .option("inferSchema", "true")
          .option("sep", ",")
          .option("escape", "\"") 
          .option("multiLine", "true")
          .load(path_completo))
    
    # Adicionando a coluna de timestamp
    df = df.withColumn("timestamp_ingestion", F.current_timestamp())
    
    # Modo Overwrite + Tolerância de Esquema
    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true") 
       .saveAsTable(f"bronze.{nome_tabela}"))
    
    print(f"Sucesso: Tabela bronze.{nome_tabela} criada com {df.count()} linhas.")

In [0]:
# 3. Lista completa de mapeamento conforme o exercício
arquivos = [
    ("olist_customers_dataset.csv", "tb_customers"),
    ("olist_geolocation_dataset.csv", "tb_geolocalizacao"),
    ("olist_order_items_dataset.csv", "tb_order_items"),
    ("olist_order_payments_dataset.csv", "tb_order_payments"),
    ("olist_order_reviews_dataset.csv", "tb_order_reviews"),
    ("olist_orders_dataset.csv", "tb_orders"),
    ("olist_products_dataset.csv", "tb_products"),
    ("olist_sellers_dataset.csv", "tb_sellers"),
    ("product_category_name_translation.csv", "tb_product_category_name_translation")
]

# Executando o loop de carga
for csv, tabela in arquivos:
    # Desabilita inferSchema para o arquivo de reviews para evitar conflito de schema
    if csv == "olist_order_reviews_dataset.csv":
        path_completo = f"{path_base}{csv}"
        print(f"Processando arquivo: {csv}")
        df = (spark.read
              .format("csv")
              .option("header", "true")
              .option("inferSchema", "false")  # Desabilitado para reviews
              .option("sep", ",")
              .option("escape", "\"")
              .option("multiLine", "true")
              .load(path_completo))
        df = df.withColumn("timestamp_ingestion", F.current_timestamp())
        df.write.format("delta").mode("overwrite").saveAsTable(f"bronze.{tabela}")
        print(f"Sucesso: Tabela bronze.{tabela} criada com {df.count()} linhas.")
    else:
        carregar_csv_para_bronze(csv, tabela)

print("\n--- CARGA DA CAMADA BRONZE CONCLUÍDA COM SUCESSO ---")

Processando arquivo: olist_customers_dataset.csv
Sucesso: Tabela bronze.tb_customers criada com 99441 linhas.
Processando arquivo: olist_geolocation_dataset.csv
Sucesso: Tabela bronze.tb_geolocalizacao criada com 1000163 linhas.
Processando arquivo: olist_order_items_dataset.csv
Sucesso: Tabela bronze.tb_order_items criada com 112650 linhas.
Processando arquivo: olist_order_payments_dataset.csv
Sucesso: Tabela bronze.tb_order_payments criada com 103886 linhas.
Processando arquivo: olist_order_reviews_dataset.csv
Sucesso: Tabela bronze.tb_order_reviews criada com 99224 linhas.
Processando arquivo: olist_orders_dataset.csv
Sucesso: Tabela bronze.tb_orders criada com 99441 linhas.
Processando arquivo: olist_products_dataset.csv
Sucesso: Tabela bronze.tb_products criada com 32951 linhas.
Processando arquivo: olist_sellers_dataset.csv
Sucesso: Tabela bronze.tb_sellers criada com 3095 linhas.
Processando arquivo: product_category_name_translation.csv
Sucesso: Tabela bronze.tb_product_categor

In [0]:
# 1. Pegando as datas reais dos pedidos na camada Bronze
datas_vendas = spark.sql("""
    SELECT 
        date_format(min(order_purchase_timestamp), 'MM-dd-yyyy') as data_ini,
        date_format(max(order_purchase_timestamp), 'MM-dd-yyyy') as data_fim
    FROM bronze.tb_orders
""").collect()

# 2. Armazenando em variáveis
data_inicio_automatica = datas_vendas[0]['data_ini']
data_fim_automatica = datas_vendas[0]['data_fim']

# 3. Atualizando os Widgets 
dbutils.widgets.text("data_inicio", data_inicio_automatica, "Data Início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", data_fim_automatica, "Data Fim (MM-DD-AAAA)")

print(f"Datas detectadas nos pedidos: Início {data_inicio_automatica} | Fim {data_fim_automatica}")

Datas detectadas nos pedidos: Início 09-04-2016 | Fim 10-17-2018


In [0]:
import requests
import pandas as pd
from pyspark.sql import functions as F

def ingestao_api_dolar_auto(data_ini, data_f):
    # Montando a URL com as variáveis automáticas
    url = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial='{data_ini}',dataFinalCotacao='{data_f}')?$select=dataHoraCotacao,cotacaoCompra&$format=json"
    
    print(f"Chamando API para o período: {data_ini} até {data_f}")
    
    response = requests.get(url)
    
    if response.status_code == 200:
        dados_json = response.json().get('value', [])
        
        if not dados_json:
            print("Atenção: A API retornou vazio para este período.")
            return

        # Criando o DataFrame
        df_pd = pd.DataFrame(dados_json)
        df_spark = spark.createDataFrame(df_pd)
        
        # Adicionando o timestamp de ingestão 
        df_spark = df_spark.withColumn("timestamp_ingestion", F.current_timestamp())
        
        # Salvando na Bronze
        df_spark.write.format("delta").mode("overwrite").saveAsTable("bronze.tb_cotacao_dolar")
        
        print(f"Sucesso! Tabela bronze.tb_cotacao_dolar atualizada.")
    else:
        print(f"Erro na API: {response.status_code}")

# Chamada da função usando as variáveis automáticas
ingestao_api_dolar_auto(data_inicio_automatica, data_fim_automatica)

Chamando API para o período: 09-04-2016 até 10-17-2018
Sucesso! Tabela bronze.tb_cotacao_dolar atualizada.


In [0]:
import requests
import pandas as pd
from pyspark.sql import functions as F

# 1. BUSCA AUTOMÁTICA DAS DATAS (Baseado nos pedidos da Bronze)
# Usamos a tb_orders para garantir que teremos dólar para todo o período de vendas
datas_vendas = spark.sql("""
    SELECT 
        date_format(min(order_purchase_timestamp), 'MM-dd-yyyy') as data_ini,
        date_format(max(order_purchase_timestamp), 'MM-dd-yyyy') as data_fim
    FROM bronze.tb_orders
""").collect()

# Atribuindo os resultados às variáveis
data_inicio_automatica = datas_vendas[0]['data_ini']
data_fim_automatica = datas_vendas[0]['data_fim']

# Isso faz com que os campos no topo do notebook mostrem as datas reais detectadas
dbutils.widgets.text("data_inicio", data_inicio_automatica, "Data Início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", data_fim_automatica, "Data Fim (MM-DD-AAAA)")

# 3. FUNÇÃO DE INGESTÃO AJUSTADA
def ingestao_api_dolar(data_ini, data_f):
    # Montando a URL com as variáveis automáticas
    url = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial='{data_ini}',dataFinalCotacao='{data_f}')?$select=dataHoraCotacao,cotacaoCompra&$format=json"
    
    print(f"Iniciando requisição ao BCB para o período: {data_ini} até {data_f}")
    
    response = requests.get(url)
    
    if response.status_code == 200:
        dados_json = response.json().get('value', [])
        
        if len(dados_json) == 0:
            print("Aviso: A API não retornou dados para este intervalo.")
            return

        # Convertendo para DataFrame
        df_pd = pd.DataFrame(dados_json)
        df_spark = spark.createDataFrame(df_pd)
        
        # Adicionando o timestamp de ingestão (Requisito Visagio)
        df_spark = df_spark.withColumn("timestamp_ingestion", F.current_timestamp())
        
        # Salvando na tabela Bronze
        df_spark.write.format("delta").mode("overwrite").saveAsTable("bronze.tb_cotacao_dolar")
        
        print(f"Sucesso! {df_spark.count()} linhas inseridas em bronze.tb_cotacao_dolar")
    else:
        print(f"Erro ao acessar API: {response.status_code} - Verifique os parâmetros.")

# 4. EXECUÇÃO UTILIZANDO AS VARIÁVEIS AUTOMÁTICAS
ingestao_api_dolar(data_inicio_automatica, data_fim_automatica)

Iniciando requisição ao BCB para o período: 09-04-2016 até 10-17-2018
Sucesso! 530 linhas inseridas em bronze.tb_cotacao_dolar
